# Projekt 2: Sprach-Werkstatt

**Teammitglieder:**  
- Gabriel Deiac
- Laurenz Berchtold

**Projektwahl:** Sprach-Werkstatt (Schreibassistenz mit HuggingFace + Gradio)

---
Dieses Notebook enthält eine lauffähige Gradio-App mit:
- Umformulieren (text-generation)
- Wort-Vorschläge mit [MASK] (fill-mask)
- Übersetzen DE↔EN, DE↔ES und DE↔IT (jeweils in beide Richtungen)
- Ton-Check (sentiment-analysis)

Zusätzlich: eigenes CSS-Design, Error Handling und History via gr.State.

## Kurzdokumentation

Die Sprach-Werkstatt unterstützt beim Schreiben durch mehrere NLP-Funktionen in einer Oberfläche.  
Für das Umformulieren verwenden wir eine text-generation Pipeline mit Stil-Auswahl (formell, informell, einfach, wissenschaftlich).  
Für Wort-Vorschläge nutzen wir fill-mask, wobei ein Satz mit [MASK] eingegeben wird und passende Wörter samt Wahrscheinlichkeit angezeigt werden.  
Die Übersetzung erfolgt mit translation zwischen Deutsch und Englisch, Spanisch sowie Italienisch in beide Richtungen.  
Zusätzlich bewertet ein Ton-Check per sentiment-analysis, ob der Text eher positiv, neutral oder negativ wirkt.  
Die App wurde mit gr.Blocks und mehreren Tabs umgesetzt.  
Ein eigenes CSS sorgt für ein konsistentes Editor/Schreibtisch-Design mit Hover-Effekten und gestylten Buttons.  
Fehleingaben wie leere oder zu kurze Texte werden abgefangen und als klare Hinweise ausgegeben.  
Als Bonus speichern wir die Umformulierungen in einer Session-History mit gr.State.

In [ ]:
import re
import pandas as pd
import gradio as gr
from transformers import pipeline

print("Lade Modelle...")

# Pipelines
text_gen = pipeline("text-generation", model="distilgpt2")
fill_mask = pipeline("fill-mask", model="bert-base-german-cased")
translator_de_en = pipeline("translation", model="Helsinki-NLP/opus-mt-de-en")
translator_en_de = pipeline("translation", model="Helsinki-NLP/opus-mt-en-de")
translator_de_es = pipeline("translation", model="Helsinki-NLP/opus-mt-de-es")
translator_es_de = pipeline("translation", model="Helsinki-NLP/opus-mt-es-de")
translator_de_it = pipeline("translation", model="Helsinki-NLP/opus-mt-de-it")
translator_it_de = pipeline("translation", model="Helsinki-NLP/opus-mt-it-de")
sentiment_pipe = pipeline("sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment")

print("Modelle geladen.")

Lade Modelle...


In [ ]:
STYLE_HINTS = {
    "formell": "Rewrite in a formal style:",
    "informell": "Rewrite in an informal style:",
    "einfach": "Rewrite in very simple language:",
    "wissenschaftlich": "Rewrite in an academic style:",
}


def _clean_text(x: str) -> str:
    return (x or "").strip()


def _short_summary(text: str, max_words: int = 20) -> str:
    words = text.split()
    if len(words) <= max_words:
        return text
    return " ".join(words[:max_words]) + " ..."


def _history_to_md(history):
    if not history:
        return "_Noch keine Umformulierungen in dieser Session._"
    lines = ["### History (Session)"]
    for i, item in enumerate(history[-10:], start=1):
        lines.append(f"{i}. **{item['style']}**  \n   ➜ {item['result']}")
    return "\n".join(lines)


def reformulate_text(text, style, history):
    text = _clean_text(text)
    history = history or []

    if not text:
        return "Bitte Text eingeben.", history, _history_to_md(history)
    if len(text) < 8:
        return "Text ist zu kurz (mindestens 8 Zeichen).", history, _history_to_md(history)

    prompt = f"{STYLE_HINTS.get(style, STYLE_HINTS['formell'])} {text}\nResult:"
    out = text_gen(
        prompt,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=50256,
    )[0]["generated_text"]

    result = out.replace(prompt, "").strip()
    if not result:
        result = out[-250:].strip()

    history.append({"style": style, "result": result})
    return result, history, _history_to_md(history)


def clear_history():
    history = []
    return history, _history_to_md(history)


def predict_mask(sentence):
    sentence = _clean_text(sentence)
    if not sentence:
        return pd.DataFrame([{"Wort": "—", "Wahrscheinlichkeit": "Bitte Satz eingeben."}])
    if "[MASK]" not in sentence:
        return pd.DataFrame([{"Wort": "—", "Wahrscheinlichkeit": "Bitte [MASK] im Satz verwenden."}])

    preds = fill_mask(sentence, top_k=5)
    rows = []
    for p in preds:
        token = p["token_str"].strip()
        prob = f"{p['score'] * 100:.2f}%"
        rows.append({"Wort": token, "Wahrscheinlichkeit": prob})
    return pd.DataFrame(rows)


def translate_text(text, direction):
    text = _clean_text(text)
    if not text:
        return "Bitte Text eingeben.", "—"

    translators = {
        "DE -> EN": translator_de_en,
        "EN -> DE": translator_en_de,
        "DE -> ES": translator_de_es,
        "ES -> DE": translator_es_de,
        "DE -> IT": translator_de_it,
        "IT -> DE": translator_it_de,
    }

    translator = translators.get(direction)
    if translator is None:
        return "Ungültige Richtung gewählt.", "—"

    translated = translator(text, max_length=256)[0]["translation_text"]
    summary = _short_summary(translated, max_words=20)
    return translated, summary


def tone_check(text):
    text = _clean_text(text)
    if not text:
        return "Bitte Text eingeben."
    if len(text) < 4:
        return "Text ist zu kurz für Ton-Analyse."

    pred = sentiment_pipe(text[:512])[0]
    label = pred["label"]
    score = pred["score"]

    mapping = {
        "positive": "Positiv",
        "negative": "Negativ",
        "neutral": "Neutral",
        "LABEL_0": "Negativ",
        "LABEL_1": "Neutral",
        "LABEL_2": "Positiv",
    }

    nice_label = mapping.get(label.lower(), mapping.get(label, label))
    return f"**Ton-Check:** {nice_label} ({score * 100:.1f}%)"

In [ ]:
custom_css = """
:root {
  --bg: #f6f1e9;
  --panel: #fffaf2;
  --accent: #8b5e3c;
  --accent-hover: #6f472c;
  --text: #2f2a24;
}
.gradio-container {
  background: var(--bg);
  color: var(--text);
}
#title-box {
  background: var(--panel);
  border: 1px solid #e4d7c8;
  border-radius: 14px;
  padding: 14px;
}
button.primary {
  background: var(--accent) !important;
  border: none !important;
  transition: transform .12s ease, background .2s ease;
}
button.primary:hover {
  background: var(--accent-hover) !important;
  transform: translateY(-1px);
}
#typewriter textarea {
  font-family: "Courier New", monospace !important;
}
"""

with gr.Blocks(css=custom_css, title="Sprach-Werkstatt") as demo:
    history_state = gr.State([])

    gr.Markdown("## Sprach-Werkstatt", elem_id="title-box")
    gr.Markdown("Schreiben, umformulieren, übersetzen und Ton prüfen - alles in einer App.")

    with gr.Row():
        with gr.Column(scale=4):
            with gr.Tabs():
                with gr.Tab("Umformulieren"):
                    with gr.Row():
                        with gr.Column(scale=2):
                            in_text = gr.Textbox(label="Text", placeholder="Gib hier deinen Text ein ...", lines=6)
                            style = gr.Dropdown(
                                choices=["formell", "informell", "einfach", "wissenschaftlich"],
                                value="formell",
                                label="Stil"
                            )
                            btn_rewrite = gr.Button("Umformulieren", variant="primary")
                            out_text = gr.Textbox(label="Ergebnis", lines=8, elem_id="typewriter")

                            btn_tone = gr.Button("Ton-Check", variant="secondary")
                            out_tone = gr.Markdown()

                        with gr.Column(scale=1):
                            hist_md = gr.Markdown("_Noch keine Umformulierungen in dieser Session._")
                            btn_clear = gr.Button("History leeren")

                    btn_rewrite.click(
                        reformulate_text,
                        inputs=[in_text, style, history_state],
                        outputs=[out_text, history_state, hist_md],
                    )
                    btn_clear.click(clear_history, outputs=[history_state, hist_md])
                    btn_tone.click(tone_check, inputs=in_text, outputs=out_tone)

                with gr.Tab("Wort-Vorschlaege ([MASK])"):
                    mask_in = gr.Textbox(
                        label="Satz mit [MASK]",
                        placeholder="Beispiel: Das Wetter ist heute [MASK].",
                        lines=3
                    )
                    mask_btn = gr.Button("Vorschlaege anzeigen", variant="primary")
                    mask_out = gr.Dataframe(label="Top-Vorschlaege")
                    mask_btn.click(predict_mask, inputs=mask_in, outputs=mask_out)

                with gr.Tab("Uebersetzer"):
                    tr_in = gr.Textbox(label="Text", placeholder="Text fuer die Uebersetzung ...", lines=5)
                    direction = gr.Radio(
                        choices=["DE -> EN", "EN -> DE", "DE -> ES", "ES -> DE", "DE -> IT", "IT -> DE"],
                        value="DE -> EN",
                        label="Richtung"
                    )
                    tr_btn = gr.Button("Uebersetzen", variant="primary")
                    tr_out = gr.Textbox(label="Uebersetzung", lines=6, elem_id="typewriter")
                    tr_sum = gr.Textbox(label="Kurzzusammenfassung", lines=2)
                    tr_btn.click(translate_text, inputs=[tr_in, direction], outputs=[tr_out, tr_sum])

        with gr.Column(scale=1, min_width=220):
            gr.Markdown("### Werkzeugleiste")
            gr.Markdown("- Umformulieren\n- Wort-Vorschlaege\n- Uebersetzen\n- Ton-Check")
            gr.Markdown("Tipp: Starte mit einem klaren Satz und pruefe danach den Ton.")

demo.launch()